# 2.0 Visualize in 2D and 3D with `CellCloud`

Using the alignment from `1.0_Preprocess_and_Align`, this notebook shows four things Celldega
does with a whole-brain serial-section dataset:

1. **Visualize in 2D:** a `Landscape` of any single slice, with a dropdown to switch slices.
2. **Visualize in 3D:** a `CellCloud` of all 3.8 million aligned cells.
3. **Compare composition across slices:** a `Composition` plot of the Leiden clusters in every
   slice, linked to the 3D view.
4. **Explore high-dimensional gene expression:** a `Clustergram` of the 480-gene signatures of
   the clusters, linked to the 3D view.

## The experiment

Whole-brain spatial transcriptomics of a **mouse embryo at E15.5** (female). The brain was cut into
**44 coronal sections**, ordered from the front of the brain (slice 1, anterior) to the back
(slice 44, posterior). The sections were placed on four Xenium slides and profiled with the
**10x Xenium** platform and its **480-gene** panel, which reads out transcripts at subcellular
resolution. Cells were segmented from the images, which gives each cell a position (in µm) and
a gene-count vector.

Consecutive slices are stacked **100 µm** apart to rebuild the brain in 3D. After quality control
the dataset holds about **3.8 million cells**.

**Using the linked widgets.** Click a cluster (a bar segment in the composition plot, or a column
in the Clustergram) to highlight it in the 3D view. Click a row of the Clustergram to color the
cells by that gene. Click a node of a dendrogram to select a group of clusters.

## Settings

In [1]:
from pathlib import Path

import anndata as ad
import json

import ipywidgets as widgets
from IPython.display import display
import numpy as np
import pandas as pd

import celldega as dega

WORKBENCH_DIR = Path("/home/jovyan/workbench")      # mirrored 1:1 to s3://$S3_BUCKET/research/projects/$PROJECT_ID/
DATA_DIR = Path("data")
DEGA_FILES_DIR = Path("DegaFiles").resolve()
CELL_CLOUD = "xenium_E15.5_cell_cloud"
LANDSCAPE_DIR = WORKBENCH_DIR / "data/DegaFiles"     # existing 2D DegaFiles, one directory per Xenium region
CLUSTER_KEY = "leiden_1.0"

# False: the bars start in slice order (anterior -> posterior), without dendrograms. True: rows and
# columns are hierarchically clustered (adds the row dendrogram; the INI button restores slice order).
COMPOSITION_CLUSTER = False
CLUSTERED_H5AD = DATA_DIR / "xenium_E15.5_clustered.h5ad"       # only needed to compute the signatures once

# Initial camera of the 3D view. rotation_x tilts from side-on (0) to top-down (90); rotation_orbit
# swings the camera around the vertical axis. Anterior is on the left.
ROTATION_X = 20
ROTATION_ORBIT = -25

import os

import boto3



def get_aws_credentials():
    creds = boto3.Session().get_credentials().get_frozen_credentials()
    return {"accessKeyId": creds.access_key, "secretAccessKey": creds.secret_key, "sessionToken": creds.token}


def dega_url(path):
    """https URL in the project bucket of a DegaFiles directory."""
    rel = Path(path).resolve().relative_to(WORKBENCH_DIR)
    return ("https://" + os.environ["S3_BUCKET"] + ".s3.us-east-1.amazonaws.com"
            "/research/projects/" + os.environ["PROJECT_ID"] + "/" + rel.as_posix())


creds = get_aws_credentials()

dega.__version__

Matplotlib is building the font cache; this may take a moment.


'0.26.1'

## 1. Visualize in 2D: a `Landscape` of one slice

A `Landscape` shows one region of tissue at full resolution: the stained image, the cell
outlines, and the transcripts and expression of every gene. Cells are colored by their Leiden
cluster. Use the dropdown to switch slices (a few regions hold several slices, which are shown
together).

In [2]:
def region_label(name):
    nums = [int(s.rstrip("b")) for s in name.split("__")[2].split("_")]
    return f"Slice {nums[0]}" if len(nums) == 1 else f"Slices {nums[0]}-{nums[-1]}"


cell_clusters = pd.read_parquet(DATA_DIR / "cell_clusters.parquet")      # cluster of every cell, from 1.0
cluster_colors = pd.read_csv(DATA_DIR / "cluster_colors.csv", index_col="cluster")["color"]
cluster_colors.index = cluster_colors.index.astype(str)
regions = sorted(cell_clusters["region"].unique(), key=lambda n: int(n.split("__")[2].split("_")[0]))


def make_landscape(region):
    # Cells of this region, named as in its DegaFiles, colored by the Leiden clusters of 1.0
    c = cell_clusters[cell_clusters["region"] == region]
    adata = ad.AnnData(obs=pd.DataFrame({CLUSTER_KEY: pd.Categorical(c[CLUSTER_KEY], categories=list(cluster_colors.index))}, index=c["barcode"].to_numpy()))
    adata.obs["cell_id"] = adata.obs_names
    adata.uns[f"{CLUSTER_KEY}_colors"] = list(cluster_colors)
    return dega.viz.Landscape(
        technology="Xenium",
        creds=creds,
        base_url=dega_url(LANDSCAPE_DIR / region),
        adata=adata,
        cell_attr=[CLUSTER_KEY],
        cluster_attr=CLUSTER_KEY,
        max_tiles_to_view=5,
        height=650,
        rotate=180,
    )


slice_picker = widgets.Dropdown(options=[(region_label(r), r) for r in regions], description="slice")
landscape_out = widgets.Output()


def show_landscape(change=None):
    with landscape_out:
        landscape_out.clear_output(wait=True)
        display(make_landscape(slice_picker.value))


slice_picker.observe(show_landscape, names="value")
show_landscape()
widgets.VBox([slice_picker, landscape_out])

## 2. Visualize in 3D: `CellCloud` linked to the cluster composition

All aligned cells in one 3D view, colored by cluster. Drag to orbit around the brain, scroll to
zoom. The composition plot beside it shows, for every slice (one bar each), the share of cells in each
cluster, so you can see which clusters belong to which part of the brain along the front-to-back
axis. Click a cluster to highlight it in 3D, or select a node of the row dendrogram to highlight a
group of related clusters (set `COMPOSITION_CLUSTER = True` to cluster the rows and columns and get the
dendrograms; the `INI` button then restores anterior-to-posterior order).

In [3]:
counts = pd.read_csv(DATA_DIR / "slice_cluster_counts.csv", index_col="slice")
cluster_colors = pd.read_csv(DATA_DIR / "cluster_colors.csv", index_col="cluster")["color"]
cluster_colors.index = cluster_colors.index.astype(str)

props = counts.div(counts.sum(axis=1), axis=0)
props.index = props.index.astype(str)
props.columns = props.columns.astype(str)
composition = dega.viz.Composition(
    props,
    colors=cluster_colors.to_dict(),
    col_weights=counts.sum(axis=1).rename(index=str).to_dict(),   # cells per slice, for the COUNTS view
    cluster=COMPOSITION_CLUSTER,
    width=520,
    height=500,
)
# Say what a row of the composition is (the "entity" of the widget, as for a Clustergram): a cluster.
# Clicking a row, or a node of the row dendrogram, then selects that cluster in the 3D view.
composition.row_entity = json.dumps({"entity": "cell_cluster", "attr": CLUSTER_KEY})

In [4]:
cell_cloud = dega.viz.CellCloud(
    creds=creds,
    base_url=dega_url(DEGA_FILES_DIR / CELL_CLOUD),
    alignment="neighborhood",
    cluster_attr=CLUSTER_KEY,
    rotation_x=ROTATION_X,
    rotation_orbit=ROTATION_ORBIT,
    height=650,
    cell_size=10,
)
cell_cloud.layout = widgets.Layout(width="55%", height="650px")
composition.layout = widgets.Layout(width="45%")
widgets.jsdlink((composition, "click_info"), (cell_cloud, "update_trigger"))   # one-way, so Python-side updates below are not echoed back
# Clicking a bar segment (a cell of the matrix) is reported as a "mat_value" click, which the 3D view does not
# act on by itself. Pass it on as a click on the cluster row, so it selects the cluster like a row click does.
def select_row_of_cell_click(change):
    info = change["new"] or {}
    if info.get("type") == "mat_value":
        cell_cloud.update_trigger = {}
        cell_cloud.update_trigger = {"type": "row_label", "value": info["value"]["row"]}


composition.observe(select_row_of_cell_click, names="click_info")
widgets.HBox([cell_cloud, composition])

The `CellCloud` holds two alignments of the same cells. Use the dropdown to flip the same view
between the centroid-only alignment (`centroids`) and the final one (`neighborhood`).

In [5]:
picker = widgets.Dropdown(options=["neighborhood", "centroids"], value="neighborhood", description="alignment")
widgets.link((picker, "value"), (cell_cloud, "alignment"))
picker

Dropdown(description='alignment', options=('neighborhood', 'centroids'), value='neighborhood')

## 3. Explore gene expression: `CellCloud` linked to a gene `Clustergram`

Each column of the Clustergram is a Leiden cluster and each row is a gene of the 480-gene panel.
Color is the row-z-scored `log1p` counts per million of the cluster, and the size of a square is
the fraction of the cluster's cells that express the gene. Rows and columns are hierarchically
clustered.

The **DIM slider** keeps only the top 1, 2, 3, 5, 10 or 25 marker genes per cluster
(`rank_genes_groups`), so you can go from a handful of markers to the whole panel. Click a gene to
color every cell in 3D by its expression, or a cluster to highlight it.

The cluster signatures are computed once from the counts of all cells and cached in `data/`.

In [6]:
SIGNATURES = DATA_DIR / "xenium_cluster_signatures.h5mu"
MARKER_LEVELS = [1, 2, 3, 5, 10, 25]

if SIGNATURES.exists():
    setc = dega.SetCollection.read(SIGNATURES)
else:
    adata = ad.read_h5ad(CLUSTERED_H5AD)
    adata.layers["counts"] = adata.X
    setc = dega.SetCollection(adata, set_col=CLUSTER_KEY, name="xenium_cluster")
    setc.calc_signature(
        adata, modality_name="expression", layer="counts", aggregate="sum", normalization="log1p_cpm",
        fraction_expressing_layer="fraction_expressing",
        rank_genes_groups=True, rank_genes_groups_kwargs={"method": "t-test"},
    )
    del adata
    # keep only the cluster-level signatures
    setc = dega.SetCollection(obs=setc.obs.copy(), mod={"expression": setc.mod["expression"].copy()}, name="xenium_cluster")
    setc.write(SIGNATURES)
setc.mod["expression"]

AnnData object with n_obs × n_vars = 38 × 480
    obs: 'leiden_1.0', 'n_cells', 'set_source', 'color'
    var: 'gene', 'leiden_1.0_marker', 'entity_type'
    uns: 'aggregate', 'axis_entities', 'expr_threshold', 'feature_type', 'fraction_expressing_layer', 'layer', 'normalization', 'rank_genes_groups'
    layers: 'fraction_expressing'

In [7]:
mat = dega.clust.Matrix(
    collection=setc,
    color_by="expression",
    size_by_layer="fraction_expressing",
    col_attr=[CLUSTER_KEY],
    global_colors=cluster_colors.to_dict(),
)
mat.norm("row", by="zscore")
mat.cluster(view="rank_genes_groups", levels=MARKER_LEVELS)       # the DIM slider
clustergram = dega.viz.Clustergram(matrix=mat, width=420, height=500, manual_col_cat=CLUSTER_KEY, rank_dim=3)

In [8]:
gene_cloud = dega.viz.CellCloud(
    creds=creds,
    base_url=dega_url(DEGA_FILES_DIR / CELL_CLOUD),
    alignment="neighborhood",
    cluster_attr=CLUSTER_KEY,
    rotation_x=ROTATION_X,
    rotation_orbit=ROTATION_ORBIT,
    height=700,
    cell_size=10,
)
dega.viz.spatial_clustergram(gene_cloud, clustergram, width="650px", height="700px")

## If a widget does not load

Right-click the page and choose **Inspect**, then right-click the browser's **Refresh** button
and choose **Empty Cache and Hard Reload**. Then re-run the widget cell.